# 日米の高収益企業の特徴比較（MSCI World 構成銘柄）

**目的**：MSCI World（先進国）構成銘柄のうち、米国（US）と日本（JP）それぞれの「高収益企業」が
どのような特徴を持つのかを洗い出し、日米の違いを整理する。

**ノートブックの構成**（上から順に実行すれば完結します。pickle は使っていません）

| # | 内容 |
|---|---|
| 0 | 設定・パラメータ |
| 1 | データ取得（`get_profitability_data`） |
| 2 | 前処理（型変換・セクター付与・外れ値処理）とデータ概観 |
| 3 | 高収益企業の定義（国×月内のROE五分位） |
| 4 | 収益性水準の推移と分布 |
| 5 | 高収益企業の特徴プロファイル（自国内での相対位置） |
| 6 | ROEの源泉：DuPont分解（マージン × 回転率 × レバレッジ） |
| 7 | セクター構成 |
| 8 | 収益性の持続性（五分位の遷移） |
| 9 | 利益の質・予想収益性 |
| 10 | 株主還元（総還元：稼いだ利益をどれだけ株主に返しているか） |
| 11 | バリュエーション（市場は収益性を評価しているか） |
| 12 | 規模・成長・投資 |
| 13 | 多変量分析（Fama-MacBeth型の月次クロスセクション回帰） |
| 14 | 頑健性（定義の変更・期間分割） |
| 15 | まとめ（自動抽出） |

**比較の考え方**
- 日米で ROE などの水準が大きく異なるため、「高収益企業」は **各国・各月のクロスセクション内での相対順位**（上位20%）で定義する。
- 特徴量も **各国・各月内で標準化（zスコア）** し、「自国の平均的な企業と比べてどれだけ偏っているか」で比べる。
  これにより、会計基準や単位の違い・市場全体の水準差の影響を抑えて「高収益企業らしさ」の日米差を見る。
- 時系列の平均と Newey-West の t 値で、偶然でない偏りかを判断する。

## 0. 設定

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import japanize_matplotlib  # noqa: F401  日本語フォント設定

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

COUNTRY_COLORS = {"US": "#1f4e9c", "JP": "#c0392b"}
COUNTRY_LABELS = {"US": "米国", "JP": "日本"}

In [ ]:
# ===== パラメータ =====
START_DATE = 200001          # 取得開始（YYYYMM）
END_DATE = 999999            # 取得終了（YYYYMM）

HIGH_PROF_VAR = "roe_actual"   # 高収益企業を定義する指標（roe_actual / roa_actual / profitability_factor など）
N_GROUPS = 5                   # 五分位
HIGH_GROUP = N_GROUPS          # 最上位グループ = 高収益企業

EXCLUDE_SECTOR_CODES = ["40"]  # 除外するGICSセクター（40=金融：ROE・レバレッジの構造が異なるため）。含める場合は []
WINSOR_LOWER, WINSOR_UPPER = 0.01, 0.99   # 国×月ごとの外れ値処理（分位点でクリップ）
MIN_SECTOR_N = 5               # セクター中立の順位付けに必要な最小銘柄数
NW_LAGS = 12                   # Newey-West t値のラグ（月次で重複する情報が多いため12）

# 期間分割（頑健性チェック用）。日本はアベノミクス／CGコード（2015）・東証PBR要請（2023）などを意識
PERIODS = {
    "2000-2012": (2000, 2012),
    "2013-2019": (2013, 2019),
    "2020-": (2020, 2100),
}

# True にすると DB に接続せず架空データで動作確認できる（分析結果には意味がない）
USE_DUMMY_DATA = False

## 1. データ取得

`fetch_data(QUERY, "irddb")` は社内DB接続用の関数です。普段お使いの import 文を下のセルに記入してください。

In [ ]:
# 社内DB接続関数の読み込み（環境に合わせて書き換えてください）
# from xxxxx import fetch_data

if not USE_DUMMY_DATA and "fetch_data" not in globals():
    raise NameError("fetch_data が定義されていません。上の import 文を環境に合わせて記入してください。")

In [ ]:
def get_profitability_data(country="US", start_date=200001, end_date=999999):
    country = country.strip().upper()

    if country not in {"US", "JP"}:
        raise ValueError("countryは'US'または'JP'を指定してください。")

    start_date = int(start_date)
    end_date = int(end_date)

    QUERY = f"""
    DECLARE @StartDate INT = {start_date};
    DECLARE @EndDate   INT = {end_date};

    SELECT
        -- 識別・管理
        F.[DATE]    AS [year_month],
        F.[GID]     AS [Global_ID],
        F.[BID]     AS [Barra_ID],
        F.[SEDOL]   AS [SEDOL],
        F.[UPDDATE] AS [update_date],

        -- MSCI銘柄情報
        NULLIF(M.[NAME], '-999999998')
            AS [msci_name],
        M.[CTRY]
            AS [msci_country],
        NULLIF(M.[GICS], '00000000')
            AS [gics_code],

        -- 企業規模
        NULLIF(F.[TMV_USD], -999999998)
            AS [company_market_cap_usd],
        NULLIF(F.[MV_USD], -999999998)
            AS [security_market_cap_usd],

        -- 実績収益性
        NULLIF(F.[ROE_ACT], -999999998)
            AS [roe_actual],
        NULLIF(F.[ROA_ACT], -999999998)
            AS [roa_actual],

        -- 予想収益性
        NULLIF(F.[ROE_EST], -999999998)
            AS [roe_estimate],
        NULLIF(F.[ROA_EST], -999999998)
            AS [roa_estimate],

        -- 収益性ファクター
        NULLIF(F.[PROF], -999999998)
            AS [profitability_factor],

        -- 収益性の源泉
        NULLIF(F.[GM_ACT], -999999998)
            AS [gross_margin_actual],
        NULLIF(F.[OIS_ACT], -999999998)
            AS [operating_margin_actual],
        NULLIF(F.[NIS_ACT], -999999998)
            AS [net_margin_actual],
        NULLIF(F.[TA_TO], -999999998)
            AS [asset_turnover_actual],

        -- 財務構成・利益の質
        NULLIF(F.[EQR], -999999998)
            AS [equity_ratio],
        NULLIF(F.[TD_TA], -999999998)
            AS [debt_to_assets],
        NULLIF(F.[OCF_TA], -999999998)
            AS [operating_cf_to_assets],

        -- バリュエーション
        NULLIF(F.[BP_ACT], -999999998)
            AS [book_to_price_actual],
        NULLIF(F.[EP_ACT], -999999998)
            AS [earnings_to_price_actual],

        -- 成長・投資
        NULLIF(F.[SALES_GRW], -999999998)
            AS [sales_growth_actual],
        NULLIF(F.[TA_GRW], -999999998)
            AS [asset_growth_actual],
        NULLIF(F.[IA], -999999998)
            AS [investment_to_assets_actual],

        -- 株主還元：配当
        NULLIF(F.[DP_ACT], -999999998)
            AS [dividend_yield_actual],
        NULLIF(F.[DP_EST], -999999998)
            AS [dividend_yield_estimate],
        NULLIF(F.[DOE], -999999998)
            AS [dividend_on_equity],

        -- 株主還元：自社株買い・総還元・純還元
        NULLIF(F.[BUYBACK], -999999998)
            AS [buyback_yield],
        NULLIF(F.[GPY], -999999998)
            AS [gross_payout_yield],
        NULLIF(F.[NPY], -999999998)
            AS [net_payout_yield]

    FROM [GLOBAL].[quants].[FACTOR] AS F

    INNER JOIN [GLOBAL].[quants].[MSCI] AS M
        ON M.[DATE] = F.[DATE]
        AND M.[GID]  = F.[GID]

    WHERE F.[DATE] BETWEEN @StartDate AND @EndDate
        AND F.[BID] <> 'XXXXXXX'
        AND M.[F_IDX_DM] = 1
        AND M.[CTRY] = '{country}'

    ORDER BY
        F.[DATE],
        F.[GID];
    """

    df = fetch_data(QUERY, "irddb")

    # 文字列の前後の空白を除去（pandas 2.1未満は applymap）
    strip = lambda x: x.strip() if isinstance(x, str) else x
    df = df.map(strip) if hasattr(df, "map") else df.applymap(strip)

    return df

In [ ]:
def make_dummy_data(country, start_date=200001, end_date=202412, n_firms=None, seed=0):
    """動作確認用の架空データ（USE_DUMMY_DATA=True のときのみ使用）。分析上の意味はない。"""
    rng = np.random.default_rng(seed + (0 if country == "US" else 1))
    n_firms = n_firms or (400 if country == "US" else 200)
    dates = pd.period_range(str(max(start_date, 200001))[:4] + "-" + str(max(start_date, 200001))[4:],
                            str(min(end_date, 202412))[:4] + "-" + str(min(end_date, 202412))[4:], freq="M")
    sectors = ["10102010", "15101010", "20101010", "25101010", "30101010", "35101010",
               "40101010", "45101010", "50101010", "55101010", "60101010"]
    firm_sector = rng.choice(sectors, n_firms)
    base_nm = rng.normal(0.08 if country == "US" else 0.05, 0.05, n_firms)
    base_to = rng.lognormal(np.log(0.7 if country == "US" else 0.8), 0.4, n_firms)
    base_eq = rng.uniform(0.2, 0.6 if country == "US" else 0.7, n_firms)
    rows = []
    nm, to, eq = base_nm.copy(), base_to.copy(), base_eq.copy()
    mcap = rng.lognormal(np.log(2e10 if country == "US" else 8e9), 1.0, n_firms)
    for d in dates:
        nm = 0.97 * nm + 0.03 * base_nm + rng.normal(0, 0.005, n_firms)
        to = to * np.exp(rng.normal(0, 0.02, n_firms))
        eq = np.clip(eq + rng.normal(0, 0.005, n_firms), 0.05, 0.95)
        mcap = mcap * np.exp(rng.normal(0.005, 0.07, n_firms))
        roa = nm * to
        roe = roa / eq
        df = pd.DataFrame({
            "year_month": int(d.strftime("%Y%m")),
            "Global_ID": [f"{country}{i:05d}" for i in range(n_firms)],
            "Barra_ID": [f"B{country}{i:04d}" for i in range(n_firms)],
            "SEDOL": None, "update_date": None,
            "msci_name": [f"{country} Corp {i}" for i in range(n_firms)],
            "msci_country": country, "gics_code": firm_sector,
            "company_market_cap_usd": mcap, "security_market_cap_usd": mcap,
            "roe_actual": roe, "roa_actual": roa,
            "roe_estimate": roe * 0.8 + 0.02 + rng.normal(0, 0.02, n_firms),
            "roa_estimate": roa * 0.8 + 0.01,
            "profitability_factor": (roe - 0.1) * 5 + rng.normal(0, 0.3, n_firms),
            "gross_margin_actual": nm + rng.uniform(0.1, 0.4, n_firms),
            "operating_margin_actual": nm * 1.4 + rng.normal(0, 0.01, n_firms),
            "net_margin_actual": nm, "asset_turnover_actual": to,
            "equity_ratio": eq, "debt_to_assets": np.clip(1 - eq - rng.uniform(0.1, 0.3, n_firms), 0, None),
            "operating_cf_to_assets": roa + rng.normal(0.03, 0.03, n_firms),
            "book_to_price_actual": rng.lognormal(np.log(0.4 if country == "US" else 0.9), 0.5, n_firms),
            "earnings_to_price_actual": rng.normal(0.05, 0.03, n_firms),
            "sales_growth_actual": rng.normal(0.05, 0.15, n_firms),
            "asset_growth_actual": rng.normal(0.05, 0.12, n_firms),
            "investment_to_assets_actual": rng.normal(0.05, 0.1, n_firms),
            "dividend_yield_actual": rng.uniform(0, 0.05, n_firms),
            "dividend_yield_estimate": rng.uniform(0, 0.05, n_firms),
            "dividend_on_equity": rng.uniform(0, 0.06, n_firms),
            "buyback_yield": rng.uniform(0, 0.04 if country == "US" else 0.015, n_firms),
            "gross_payout_yield": rng.uniform(0, 0.08, n_firms),
            "net_payout_yield": rng.uniform(-0.02, 0.07, n_firms),
        })
        rows.append(df)
    return pd.concat(rows, ignore_index=True)


def load_data(country):
    if USE_DUMMY_DATA:
        return make_dummy_data(country, START_DATE, END_DATE)
    return get_profitability_data(country, START_DATE, END_DATE)

In [ ]:
df_us = load_data("US")
df_jp = load_data("JP")

raw = pd.concat([df_us.assign(country="US"), df_jp.assign(country="JP")], ignore_index=True)
print(f"US: {len(df_us):,} 行 / JP: {len(df_jp):,} 行")
raw.head()

## 2. 前処理とデータ概観

- `year_month`（YYYYMM）を日付に変換
- 数値列を数値型へ（念のため欠損コード `-999999998` 系も欠損に）
- GICS コード先頭2桁からセクターを付与し、指定セクター（既定：金融）を除外
- 国×月ごとに 1%/99% でウィンザー化

In [ ]:
GICS_SECTOR = {
    "10": "エネルギー", "15": "素材", "20": "資本財・サービス", "25": "一般消費財・サービス",
    "30": "生活必需品", "35": "ヘルスケア", "40": "金融", "45": "情報技術",
    "50": "コミュニケーション・サービス", "55": "公益事業", "60": "不動産",
}

# 分析に使う数値列と日本語ラベル
LABELS = {
    "roe_actual": "ROE（実績）",
    "roa_actual": "ROA（実績）",
    "roe_estimate": "ROE（予想）",
    "roa_estimate": "ROA（予想）",
    "profitability_factor": "収益性ファクター",
    "gross_margin_actual": "売上総利益率",
    "operating_margin_actual": "営業利益率",
    "net_margin_actual": "純利益率",
    "asset_turnover_actual": "総資産回転率",
    "equity_ratio": "自己資本比率",
    "debt_to_assets": "有利子負債/総資産",
    "operating_cf_to_assets": "営業CF/総資産",
    "book_to_price_actual": "B/P",
    "earnings_to_price_actual": "E/P",
    "sales_growth_actual": "売上成長率",
    "asset_growth_actual": "総資産成長率",
    "investment_to_assets_actual": "投資/総資産",
    "dividend_yield_actual": "配当利回り（実績）",
    "dividend_yield_estimate": "配当利回り（予想）",
    "dividend_on_equity": "DOE",
    "buyback_yield": "自社株買い利回り",
    "gross_payout_yield": "総還元利回り",
    "net_payout_yield": "純還元利回り",
    "company_market_cap_usd": "時価総額（企業, USD）",
    "security_market_cap_usd": "時価総額（銘柄, USD）",
    # 派生変数
    "log_mcap": "log時価総額",
    "total_payout_ratio": "総還元性向（総還元利回り÷E/P）",
    "net_payout_ratio": "純還元性向（純還元利回り÷E/P）",
    "accrual": "アクルーアル（ROA−営業CF/総資産）",
    "roe_change_est": "予想ROE−実績ROE",
}
NUM_COLS = [c for c in LABELS if c in raw.columns]
lab = lambda c: LABELS.get(c, c)

In [ ]:
df = raw.copy()
df["date"] = pd.to_datetime(df["year_month"].astype(int).astype(str), format="%Y%m")

for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df[NUM_COLS] = df[NUM_COLS].mask(df[NUM_COLS] <= -999999990)   # 欠損コードの取りこぼし対策

# GICS（数値で返ってきても文字列で返ってきても対応）
gics = pd.to_numeric(df["gics_code"], errors="coerce").astype("Int64").astype("string")
df["sector_code"] = gics.str[:2]
df["sector"] = df["sector_code"].map(GICS_SECTOR).fillna("不明")

# 重複（同一国・同一月・同一GID）は最初の1行を残す
n0 = len(df)
df = df.drop_duplicates(["country", "date", "Global_ID"]).copy()
print(f"重複除去: {n0 - len(df):,} 行")

# セクター除外
n0 = len(df)
df = df[~df["sector_code"].isin(EXCLUDE_SECTOR_CODES)].copy()
print(f"セクター除外 {EXCLUDE_SECTOR_CODES}: {n0 - len(df):,} 行")

# 派生変数
df["log_mcap"] = np.log(df["company_market_cap_usd"].where(df["company_market_cap_usd"] > 0))
df["accrual"] = df["roa_actual"] - df["operating_cf_to_assets"]
df["roe_change_est"] = df["roe_estimate"] - df["roe_actual"]
# 総還元性向 = (配当+自社株買い)/純利益 = 総還元利回り / 益利回り（E/P>0 の企業のみ。赤字企業は定義できない）
ep_pos = df["earnings_to_price_actual"].where(df["earnings_to_price_actual"] > 0)
df["total_payout_ratio"] = df["gross_payout_yield"] / ep_pos
df["net_payout_ratio"] = df["net_payout_yield"] / ep_pos

ANALYSIS_COLS = [c for c in NUM_COLS + ["log_mcap", "accrual", "roe_change_est",
                                        "total_payout_ratio", "net_payout_ratio"] if c in df.columns]

In [ ]:
def winsorize_by_group(data, cols, keys, lower=0.01, upper=0.99):
    """keys（国×月など）ごとに分位点でクリップする。"""
    g = data.groupby(keys)[cols]
    lo = data[keys].join(g.quantile(lower), on=keys)[cols]
    hi = data[keys].join(g.quantile(upper), on=keys)[cols]
    out = data.copy()
    out[cols] = data[cols].clip(lower=lo, upper=hi)
    return out


df = winsorize_by_group(df, ANALYSIS_COLS, ["country", "date"], WINSOR_LOWER, WINSOR_UPPER)
df = df.sort_values(["country", "date", "Global_ID"]).reset_index(drop=True)
print(f"分析対象: {len(df):,} 行, 期間 {df['date'].min():%Y-%m} 〜 {df['date'].max():%Y-%m}")

### 単位の確認
DB の値が「%表記」か「小数表記」かで解釈が変わるため、まず国別の中央値を確認します
（以降の分析は国内標準化・順位・対数差を中心にしているので単位に依存しにくい設計です）。

In [ ]:
summary_stats = (
    df.groupby("country")[ANALYSIS_COLS]
    .median()
    .T.rename(index=LABELS)
    .rename(columns=COUNTRY_LABELS)
)
summary_stats.columns.name = "中央値（全期間プール）"
summary_stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

cnt = df.groupby(["date", "country"])["Global_ID"].nunique().unstack()
for c in ["US", "JP"]:
    axes[0].plot(cnt.index, cnt[c], color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
axes[0].set_title("銘柄数の推移（金融除外後）")
axes[0].legend()

miss = df.groupby("country")[ANALYSIS_COLS].apply(lambda x: x.isna().mean()).T
miss = miss.rename(index=LABELS)
x = np.arange(len(miss))
axes[1].bar(x - 0.2, miss["US"], 0.4, color=COUNTRY_COLORS["US"], label="米国")
axes[1].bar(x + 0.2, miss["JP"], 0.4, color=COUNTRY_COLORS["JP"], label="日本")
axes[1].set_xticks(x)
axes[1].set_xticklabels(miss.index, rotation=90, fontsize=8)
axes[1].set_title("欠損率（全期間）")
axes[1].legend()
plt.tight_layout()
plt.show()

sec_cnt = (df.groupby(["country", "sector"])["Global_ID"].nunique().unstack(0)
           .rename(columns=COUNTRY_LABELS).fillna(0).astype(int))
sec_cnt.columns.name = "ユニーク銘柄数"
sec_cnt

## 3. 高収益企業の定義

- **基本定義**：各国・各月のクロスセクションで `HIGH_PROF_VAR`（既定 ROE実績）の五分位を作り、第5分位を「高収益企業」とする。
- **セクター中立定義**：国×月×セクター内で五分位を作る（業種構成の違いを除いた「同業内で稼いでいる企業」）。
- **日米共通基準**：日米を合算して順位付けした場合の第5分位（絶対水準での比較用）。

In [ ]:
def assign_groups(data, var, keys, n=N_GROUPS, min_n=1):
    r = data.groupby(keys)[var].rank(pct=True)
    size = data.groupby(keys)[var].transform("count")
    q = np.ceil(r * n).clip(1, n)
    return q.where(size >= min_n)


def add_group_columns(data, var):
    out = data.copy()
    out["q"] = assign_groups(out, var, ["country", "date"])
    out["q_sector"] = assign_groups(out, var, ["country", "date", "sector"], min_n=MIN_SECTOR_N)
    out["q_global"] = assign_groups(out, var, ["date"])
    out["is_high"] = (out["q"] == HIGH_GROUP).where(out["q"].notna())
    out["is_high_sector"] = (out["q_sector"] == HIGH_GROUP).where(out["q_sector"].notna())
    out["is_high_global"] = (out["q_global"] == HIGH_GROUP).where(out["q_global"].notna())
    out["group"] = np.where(out["q"] == HIGH_GROUP, "高収益(Q5)", np.where(out["q"].notna(), "その他(Q1-4)", None))
    return out


df = add_group_columns(df, HIGH_PROF_VAR)
df.groupby("country")[["q", "q_sector", "q_global"]].count().rename(index=COUNTRY_LABELS)

In [ ]:
# 日米共通基準での「高収益企業」に占める日本企業の割合と、各国内での比率
glob = df.dropna(subset=["q_global"]).groupby(["date", "country"])["is_high_global"].mean().unstack()

fig, ax = plt.subplots(figsize=(12, 4))
for c in ["US", "JP"]:
    ax.plot(glob.index, glob[c], color=COUNTRY_COLORS[c], label=f"{COUNTRY_LABELS[c]}企業のうち共通基準Q5に入る割合")
ax.axhline(0.2, color="gray", ls="--", lw=1, label="均等なら20%")
ax.set_title(f"日米合算で順位付けした場合の高収益企業比率（{lab(HIGH_PROF_VAR)}）")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.legend()
plt.show()

## 4. 収益性水準の推移と分布

In [ ]:
def plot_country_median_ts(data, cols, ncols=3, title=None):
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.3 * nrows), squeeze=False)
    med = data.groupby(["date", "country"])[cols].median()
    for ax, col in zip(axes.flat, cols):
        m = med[col].unstack()
        for c in ["US", "JP"]:
            if c in m:
                ax.plot(m.index, m[c], color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
        ax.set_title(lab(col))
    for ax in axes.flat[len(cols):]:
        ax.set_visible(False)
    axes.flat[0].legend()
    if title:
        fig.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.show()


plot_country_median_ts(
    df,
    ["roe_actual", "roa_actual", "profitability_factor",
     "operating_margin_actual", "net_margin_actual", "asset_turnover_actual"],
    title="収益性指標の中央値の推移",
)

In [ ]:
# 高収益企業（Q5）の下限値（＝閾値）と中央値の推移
th = (df.dropna(subset=["q"]).groupby(["date", "country"])[HIGH_PROF_VAR]
      .quantile(1 - 1 / N_GROUPS).unstack())
q5_med = df[df["is_high"] == True].groupby(["date", "country"])[HIGH_PROF_VAR].median().unstack()

fig, ax = plt.subplots(figsize=(12, 4))
for c in ["US", "JP"]:
    ax.plot(th.index, th[c], color=COUNTRY_COLORS[c], ls="--", label=f"{COUNTRY_LABELS[c]}：Q5の閾値")
    ax.plot(q5_med.index, q5_med[c], color=COUNTRY_COLORS[c], label=f"{COUNTRY_LABELS[c]}：Q5の中央値")
ax.set_title(f"高収益企業の{lab(HIGH_PROF_VAR)}水準")
ax.legend(ncol=2)
plt.show()

In [ ]:
# 直近時点の分布
latest = df["date"].max()
snap = df[df["date"] == latest]
cols = ["roe_actual", "roa_actual", "operating_margin_actual", "asset_turnover_actual", "equity_ratio", "book_to_price_actual"]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.flat, cols):
    for c in ["US", "JP"]:
        v = snap.loc[snap["country"] == c, col].dropna()
        ax.hist(v, bins=40, alpha=0.5, density=True, color=COUNTRY_COLORS[c], label=f"{COUNTRY_LABELS[c]} (n={len(v)})")
        ax.axvline(v.median(), color=COUNTRY_COLORS[c], ls="--", lw=1)
    ax.set_title(lab(col))
axes.flat[0].legend()
fig.suptitle(f"分布の比較（{latest:%Y-%m}時点、破線は中央値）", y=1.01)
plt.tight_layout()
plt.show()

## 5. 高収益企業の特徴プロファイル

各国・各月で特徴量を標準化（zスコア）し、高収益企業（Q5）の平均zスコアを月ごとに計算 → 全期間で平均します。

- 値が **+** なら「自国の平均的企業より高い」、**−** なら「低い」
- t値は Newey-West（ラグ12）。|t| > 2 程度を目安に「一貫した偏り」とみなす
- 日米の差（JP − US）が大きい項目が、**日米で高収益企業の性格が異なるポイント**

※ ROE を定義に使っているので、ROE・ROA・マージン等が高いのは当然。注目すべきは
**財務構成・還元・バリュエーション・成長・規模** といった「定義に直接含まれない」項目と、ROEの各構成要素のバランス。

In [ ]:
PROFILE_FEATS = [c for c in [
    "roe_actual", "roa_actual", "roe_estimate", "profitability_factor",
    "gross_margin_actual", "operating_margin_actual", "net_margin_actual", "asset_turnover_actual",
    "equity_ratio", "debt_to_assets", "operating_cf_to_assets", "accrual",
    "book_to_price_actual", "earnings_to_price_actual",
    "sales_growth_actual", "asset_growth_actual", "investment_to_assets_actual",
    "gross_payout_yield", "net_payout_yield", "total_payout_ratio", "net_payout_ratio",
    "log_mcap",
] if c in df.columns]

FEATURE_BLOCK = {
    "収益性": ["roe_actual", "roa_actual", "roe_estimate", "profitability_factor"],
    "収益の源泉": ["gross_margin_actual", "operating_margin_actual", "net_margin_actual", "asset_turnover_actual"],
    "財務構成・利益の質": ["equity_ratio", "debt_to_assets", "operating_cf_to_assets", "accrual"],
    "バリュエーション": ["book_to_price_actual", "earnings_to_price_actual"],
    "成長・投資": ["sales_growth_actual", "asset_growth_actual", "investment_to_assets_actual"],
    "株主還元（総還元）": ["gross_payout_yield", "net_payout_yield", "total_payout_ratio", "net_payout_ratio"],
    "規模": ["log_mcap"],
}


def nw_tstat(x, lags=NW_LAGS):
    """時系列平均の Newey-West t値"""
    x = pd.Series(x).dropna().to_numpy()
    n = len(x)
    if n < 3:
        return np.nan
    e = x - x.mean()
    s = e @ e / n
    for l in range(1, min(lags, n - 1) + 1):
        s += 2 * (1 - l / (lags + 1)) * (e[l:] @ e[:-l]) / n
    return x.mean() / np.sqrt(s / n) if s > 0 else np.nan


def cs_zscore(data, cols, keys=("country", "date")):
    keys = list(keys)
    g = data.groupby(keys)[cols]
    return (data[cols] - g.transform("mean")) / g.transform("std")


def high_group_profile(data, flag_col="is_high", feats=PROFILE_FEATS):
    """国×月ごとの『高収益企業の平均zスコア』の時系列と、その平均・t値を返す"""
    z = cs_zscore(data, feats)
    z[["country", "date"]] = data[["country", "date"]]
    ts = z[data[flag_col] == True].groupby(["country", "date"])[feats].mean()
    mean = ts.groupby("country").mean().T
    tval = ts.groupby("country").agg(lambda s: nw_tstat(s)).T
    res = pd.DataFrame({
        "US_z": mean.get("US"), "JP_z": mean.get("JP"),
        "US_t": tval.get("US"), "JP_t": tval.get("JP"),
    })
    res["JP−US"] = res["JP_z"] - res["US_z"]
    return res, ts


profile, profile_ts = high_group_profile(df)
profile.rename(index=LABELS).style.format("{:.2f}").background_gradient(
    cmap="RdBu_r", subset=["US_z", "JP_z", "JP−US"], vmin=-1.2, vmax=1.2)

In [ ]:
def plot_profile_bars(res, title, sort_by="JP−US"):
    r = res.sort_values(sort_by)
    y = np.arange(len(r))
    fig, ax = plt.subplots(figsize=(9, 0.38 * len(r) + 1.5))
    ax.barh(y + 0.2, r["US_z"], 0.4, color=COUNTRY_COLORS["US"], label="米国")
    ax.barh(y - 0.2, r["JP_z"], 0.4, color=COUNTRY_COLORS["JP"], label="日本")
    for i, (_, row) in enumerate(r.iterrows()):
        for c, off in (("US", 0.2), ("JP", -0.2)):
            if abs(row[f"{c}_t"]) < 2:   # 有意でないものはハッチで表示
                ax.barh(i + off, row[f"{c}_z"], 0.4, fill=False, hatch="///", edgecolor="white", lw=0)
    ax.set_yticks(y)
    ax.set_yticklabels([lab(i) for i in r.index])
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("高収益企業の平均zスコア（自国内で標準化）")
    ax.set_title(title + "\n（斜線は|t|<2）")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


plot_profile_bars(profile, f"高収益企業（{lab(HIGH_PROF_VAR)} Q5）の特徴：日米比較（JP−US 差の順）")

In [ ]:
# 差（JP − US）だけを見る：右ほど「日本の高収益企業でより強く見られる特徴」
r = profile.drop(index=[c for c in FEATURE_BLOCK["収益性"] if c in profile.index]).sort_values("JP−US")
fig, ax = plt.subplots(figsize=(9, 0.35 * len(r) + 1.5))
ax.barh([lab(i) for i in r.index], r["JP−US"],
        color=np.where(r["JP−US"] > 0, COUNTRY_COLORS["JP"], COUNTRY_COLORS["US"]))
ax.axvline(0, color="black", lw=0.8)
ax.set_title("高収益企業の特徴の日米差（JP−US, zスコア）\n赤：日本の高収益企業でより顕著 / 青：米国の高収益企業でより顕著")
plt.tight_layout()
plt.show()

In [ ]:
# 特徴の偏りは時間とともに変化しているか（主要項目、12か月移動平均）
ts_cols = [c for c in ["equity_ratio", "asset_turnover_actual", "operating_margin_actual",
                       "book_to_price_actual", "total_payout_ratio", "log_mcap"] if c in profile_ts.columns]
fig, axes = plt.subplots(2, 3, figsize=(15, 6.5))
for ax, col in zip(axes.flat, ts_cols):
    for c in ["US", "JP"]:
        s = profile_ts.loc[c, col].rolling(12, min_periods=6).mean()
        ax.plot(s.index, s, color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"高収益企業の{lab(col)}（z, 12M平均）")
axes.flat[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# 絶対水準での比較：高収益企業 vs その他 の中央値（月次中央値の時系列平均）
level = (df.dropna(subset=["group"])
         .groupby(["country", "date", "group"])[PROFILE_FEATS].median()
         .groupby(["country", "group"]).mean().T)
level = level[[("US", "高収益(Q5)"), ("US", "その他(Q1-4)"), ("JP", "高収益(Q5)"), ("JP", "その他(Q1-4)")]]
level.index = [lab(i) for i in level.index]
level.columns = [f"{COUNTRY_LABELS[c]}:{g}" for c, g in level.columns]
level

## 6. ROE の源泉：DuPont 分解

$$\text{ROE} = \underbrace{\frac{\text{純利益}}{\text{売上}}}_{\text{純利益率}} \times \underbrace{\frac{\text{売上}}{\text{総資産}}}_{\text{回転率}} \times \underbrace{\frac{\text{総資産}}{\text{自己資本}}}_{\text{レバレッジ = 1/自己資本比率}}$$

対数を取ると和になるので、各国・各月のクロスセクションで

$$\mathrm{Var}(\log \text{ROE}) \approx \mathrm{Cov}(\log \text{ROE}, \log \text{NM}) + \mathrm{Cov}(\log \text{ROE}, \log \text{TO}) + \mathrm{Cov}(\log \text{ROE}, \log \text{LEV})$$

と分解し、**「ROE の企業間格差のうち何割がマージン/回転率/レバレッジで説明されるか」** を日米で比べます。
対数を使うため単位（%か小数か）に依存しません。ROE・純利益率などが正の企業のみが対象です。

In [ ]:
dp = df[["country", "date", "Global_ID", "q", "is_high", "roe_actual", "net_margin_actual",
         "asset_turnover_actual", "equity_ratio"]].copy()
pos = (dp[["roe_actual", "net_margin_actual", "asset_turnover_actual", "equity_ratio"]] > 0).all(axis=1)
dp = dp[pos].copy()
dp["l_roe"] = np.log(dp["roe_actual"])
dp["l_nm"] = np.log(dp["net_margin_actual"])
dp["l_to"] = np.log(dp["asset_turnover_actual"])
dp["l_lev"] = -np.log(dp["equity_ratio"])
COMP = {"l_nm": "純利益率", "l_to": "総資産回転率", "l_lev": "レバレッジ"}

# 分解の当てはまり（定義が一致していれば相関≒1）
dp["l_sum"] = dp[list(COMP)].sum(axis=1)
fit = dp.groupby("country")[["l_roe", "l_sum"]].corr().xs("l_roe", level=1)["l_sum"]
print("log ROE と log(NM×TO×LEV) の相関:", fit.round(3).to_dict(),
      "（1から大きく離れる場合、ROEの定義（平均資本など）が構成要素と異なる）")

# 国×月ごとの分散分解
g = dp.groupby(["country", "date"])
var_roe = g["l_roe"].var()
shares = pd.DataFrame({
    name: g.apply(lambda x, c=c: x["l_roe"].cov(x[c])) / var_roe for c, name in COMP.items()
})
shares["その他（近似誤差）"] = 1 - shares.sum(axis=1)
share_mean = shares.groupby("country").mean().rename(index=COUNTRY_LABELS)
share_mean

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1, 2]})

colors = ["#2a9d8f", "#e9c46a", "#e76f51", "#bbbbbb"]
left = np.zeros(len(share_mean))
for col, colr in zip(share_mean.columns, colors):
    axes[0].barh(share_mean.index, share_mean[col], left=left, color=colr, label=col)
    left += share_mean[col].values
axes[0].set_title("ROE格差の寄与度（全期間平均）")
axes[0].xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axes[0].legend(fontsize=8, loc="lower right")

for c in ["US", "JP"]:
    for name, ls in zip(COMP.values(), ["-", "--", ":"]):
        s = shares.loc[c, name].rolling(12, min_periods=6).mean()
        axes[1].plot(s.index, s, color=COUNTRY_COLORS[c], ls=ls, label=f"{COUNTRY_LABELS[c]}:{name}")
axes[1].set_title("寄与度の推移（12M平均）")
axes[1].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axes[1].legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 高収益企業（Q5）とその他の構成要素の差（対数差 → 倍率）
diff = (dp.dropna(subset=["q"])
        .assign(grp=lambda x: np.where(x["is_high"] == True, "Q5", "Q1-4"))
        .groupby(["country", "date", "grp"])[list(COMP)].mean()
        .unstack("grp"))
diff = (diff.xs("Q5", axis=1, level=1) - diff.xs("Q1-4", axis=1, level=1)).groupby("country").mean()
ratio = np.exp(diff).rename(columns=COMP, index=COUNTRY_LABELS)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(COMP))
for i, c in enumerate(["米国", "日本"]):
    ax.bar(x + (i - 0.5) * 0.38, ratio.loc[c], 0.38, color=list(COUNTRY_COLORS.values())[i], label=c)
    for xi, v in zip(x, ratio.loc[c]):
        ax.text(xi + (i - 0.5) * 0.38, v, f"{v:.2f}倍", ha="center", va="bottom", fontsize=9)
ax.axhline(1, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(COMP.values())
ax.set_ylabel("高収益企業 ÷ その他（幾何平均の比）")
ax.set_title("高収益企業は何で稼いでいるか（DuPont構成要素の倍率）")
ax.legend()
plt.show()
ratio

In [ ]:
# マージン × 回転率 マップ（直近時点）
snap = df[(df["date"] == latest)].dropna(subset=["operating_margin_actual", "asset_turnover_actual", "q"])
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True, sharey=True)
for ax, c in zip(axes, ["US", "JP"]):
    s = snap[snap["country"] == c]
    o = s[s["is_high"] != True]
    h = s[s["is_high"] == True]
    ax.scatter(o["asset_turnover_actual"], o["operating_margin_actual"], s=10, color="lightgray", label="その他")
    ax.scatter(h["asset_turnover_actual"], h["operating_margin_actual"], s=18, color=COUNTRY_COLORS[c], label="高収益(Q5)")
    ax.set_xscale("log")
    ax.set_title(f"{COUNTRY_LABELS[c]}（{latest:%Y-%m}）")
    ax.set_xlabel("総資産回転率（対数軸）")
    ax.set_ylabel("営業利益率")
    ax.legend()
fig.suptitle("営業利益率 × 総資産回転率：高収益企業の位置", y=1.01)
plt.tight_layout()
plt.show()

## 7. セクター構成

高収益企業がどの業種に偏っているかを日米で比較します。
「ユニバース構成比」と「高収益企業内の構成比」の差が大きい業種ほど、その国では業種要因で高収益になりやすい。

In [ ]:
sec = (df.dropna(subset=["q"]).groupby(["country", "date", "sector"])["is_high"]
       .agg(["sum", "count"]).reset_index())
sec["share_univ"] = sec["count"] / sec.groupby(["country", "date"])["count"].transform("sum")
sec["share_high"] = sec["sum"] / sec.groupby(["country", "date"])["sum"].transform("sum")
sec["hit_rate"] = sec["sum"] / sec["count"]
sec_avg = sec.groupby(["country", "sector"])[["share_univ", "share_high", "hit_rate"]].mean()

sectors = sorted(df["sector"].unique(), key=lambda s: sec_avg.xs(s, level="sector")["share_high"].mean()
                 if s in sec_avg.index.get_level_values("sector") else 0)
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
y = np.arange(len(sectors))
for ax, c in zip(axes[:2], ["US", "JP"]):
    s = sec_avg.loc[c].reindex(sectors)
    ax.barh(y + 0.2, s["share_univ"], 0.4, color="lightgray", label="ユニバース")
    ax.barh(y - 0.2, s["share_high"], 0.4, color=COUNTRY_COLORS[c], label="高収益企業")
    ax.set_title(f"{COUNTRY_LABELS[c]}：セクター構成比")
    ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    ax.legend()
ax = axes[2]
for i, c in enumerate(["US", "JP"]):
    s = sec_avg.loc[c].reindex(sectors)
    ax.barh(y + (0.2 if c == "US" else -0.2), s["hit_rate"], 0.4, color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
ax.axvline(1 / N_GROUPS, color="black", ls="--", lw=0.8)
ax.set_title("セクター内で高収益企業に入る確率")
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.legend()
axes[0].set_yticks(y)
axes[0].set_yticklabels(sectors)
plt.tight_layout()
plt.show()

In [ ]:
# セクター中立定義でのプロファイル：業種構成の違いを除いても日米差は残るか
profile_sec, _ = high_group_profile(df, "is_high_sector")
cmp = pd.DataFrame({
    "JP−US（基本）": profile["JP−US"],
    "JP−US（セクター中立）": profile_sec["JP−US"],
}).drop(index=[c for c in FEATURE_BLOCK["収益性"] if c in profile.index])
cmp = cmp.sort_values("JP−US（基本）")

fig, ax = plt.subplots(figsize=(9, 0.38 * len(cmp) + 1.5))
yy = np.arange(len(cmp))
ax.barh(yy + 0.2, cmp.iloc[:, 0], 0.4, color="#555555", label=cmp.columns[0])
ax.barh(yy - 0.2, cmp.iloc[:, 1], 0.4, color="#e9a23b", label=cmp.columns[1])
ax.set_yticks(yy)
ax.set_yticklabels([lab(i) for i in cmp.index])
ax.axvline(0, color="black", lw=0.8)
ax.set_title("高収益企業の特徴の日米差：業種構成をコントロールしても残るか")
ax.legend()
plt.tight_layout()
plt.show()

## 8. 収益性の持続性

高収益企業は翌年以降も高収益であり続けるか（＝競争優位の持続性）を、五分位の遷移で比較します。
ランダムなら Q5 に残る確率は 20% です。

In [ ]:
panel = df[["country", "Global_ID", "date", "q"]].dropna()


def forward_groups(p, months):
    f = p.copy()
    f["date"] = f["date"] - pd.DateOffset(months=months)
    return p.merge(f.rename(columns={"q": "q_fwd"}), on=["country", "Global_ID", "date"], how="inner")


horizons = [12, 24, 36, 48, 60]
stay = {}
for h in horizons:
    m = forward_groups(panel, h)
    stay[h] = m[m["q"] == HIGH_GROUP].groupby(["country", "date"]).apply(
        lambda x: (x["q_fwd"] == HIGH_GROUP).mean()).groupby("country").mean()
stay = pd.DataFrame(stay).T

trans12 = forward_groups(panel, 12)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for c in ["US", "JP"]:
    axes[0].plot(stay.index, stay[c], marker="o", color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
axes[0].axhline(1 / N_GROUPS, color="gray", ls="--", lw=1, label="ランダム")
axes[0].set_xlabel("経過月数")
axes[0].set_title("高収益企業(Q5)がQ5に留まる確率")
axes[0].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axes[0].legend()

for ax, c in zip(axes[1:], ["US", "JP"]):
    t = pd.crosstab(trans12.loc[trans12.country == c, "q"], trans12.loc[trans12.country == c, "q_fwd"], normalize="index")
    im = ax.imshow(t.values, cmap="Blues", vmin=0, vmax=0.8)
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            ax.text(j, i, f"{t.values[i, j]:.0%}", ha="center", va="center",
                    color="white" if t.values[i, j] > 0.45 else "black", fontsize=9)
    ax.set_xticks(range(N_GROUPS)); ax.set_xticklabels([f"Q{i}" for i in range(1, N_GROUPS + 1)])
    ax.set_yticks(range(N_GROUPS)); ax.set_yticklabels([f"Q{i}" for i in range(1, N_GROUPS + 1)])
    ax.set_xlabel("12か月後"); ax.set_ylabel("当月")
    ax.set_title(f"{COUNTRY_LABELS[c]}：12か月後の遷移確率")
    ax.grid(False)
plt.tight_layout()
plt.show()
stay.rename(columns=COUNTRY_LABELS)

In [ ]:
# 持続性の時系列（12か月後もQ5に留まる確率、12M平均）
p12 = trans12[trans12["q"] == HIGH_GROUP].groupby(["date", "country"]).apply(
    lambda x: (x["q_fwd"] == HIGH_GROUP).mean()).unstack()
fig, ax = plt.subplots(figsize=(12, 3.8))
for c in ["US", "JP"]:
    ax.plot(p12.index, p12[c].rolling(12, min_periods=6).mean(), color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
ax.set_title("12か月後もQ5に留まる確率の推移（12M平均）")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.legend()
plt.show()

## 9〜12. 五分位別のプロファイル

ROE五分位（Q1→Q5）ごとに各指標の中央値（月次中央値の時系列平均）を描き、
**高収益になるほど何が上がり/下がるのか、その傾きが日米でどう違うか** を見ます。

In [ ]:
def plot_quintile_profiles(data, cols, title, ncols=3):
    cols = [c for c in cols if c in data.columns]
    prof = (data.dropna(subset=["q"]).groupby(["country", "date", "q"])[cols].median()
            .groupby(["country", "q"]).mean())
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 3.3 * nrows), squeeze=False)
    for ax, col in zip(axes.flat, cols):
        for c in ["US", "JP"]:
            s = prof.loc[c, col]
            ax.plot(s.index, s.values, marker="o", color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
        ax.set_xticks(range(1, N_GROUPS + 1))
        ax.set_xticklabels([f"Q{i}" for i in range(1, N_GROUPS + 1)])
        ax.set_title(lab(col))
    for ax in axes.flat[len(cols):]:
        ax.set_visible(False)
    axes.flat[0].legend()
    fig.suptitle(f"{title}（{lab(HIGH_PROF_VAR)}五分位別の中央値）", y=1.01)
    plt.tight_layout()
    plt.show()
    out = prof[cols].unstack("country")
    out.columns = [f"{lab(a)}:{COUNTRY_LABELS[b]}" for a, b in out.columns]
    return out

### 9. 利益の質・予想収益性

In [ ]:
_ = plot_quintile_profiles(
    df,
    ["operating_cf_to_assets", "accrual", "roa_actual", "roe_estimate", "roe_change_est", "profitability_factor"],
    "利益の質・予想収益性",
)

- **アクルーアル**（ROA − 営業CF/総資産）が高い＝利益がキャッシュを伴っていない。高収益企業でこれが高い国は「利益の質」に注意。
- **予想ROE − 実績ROE** が Q5 でマイナス＝アナリストが平均回帰を予想。日米で回帰の強さが違うかを確認。

### 10. 株主還元：稼いだ利益をどれだけ株主に返しているか

配当か自社株買いかの手段は問わず、**総還元**で見ます。

- **総還元利回り**（配当＋自社株買い）÷ 時価総額：投資家から見た還元の大きさ
- **総還元性向** ＝ 総還元利回り ÷ E/P ＝（配当＋自社株買い）÷ 純利益：**稼いだ利益のうち何割を返しているか**
- **純還元**は株式発行（希薄化）を差し引いたもの

総還元性向は E/P>0（黒字）の企業のみで計算しています。100%超は利益以上を還元（手元資金の取り崩しや借入）していることを表します。

In [ ]:
_ = plot_quintile_profiles(
    df,
    ["total_payout_ratio", "net_payout_ratio", "gross_payout_yield", "net_payout_yield"],
    "総還元", ncols=4,
)

In [ ]:
# 高収益企業の総還元性向の分布：日米の高収益企業は利益の何割を返しているか（全期間プール）
hp = df[(df["is_high"] == True)].dropna(subset=["total_payout_ratio"])
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
data_box = [hp.loc[hp["country"] == c, "total_payout_ratio"] for c in ["US", "JP"]]
axes[0].boxplot(data_box, showfliers=False, widths=0.5)
axes[0].set_xticks([1, 2])
axes[0].set_xticklabels(["米国", "日本"])
axes[0].axhline(hp["total_payout_ratio"].median(), color="gray", ls=":", lw=1)
axes[0].set_title("高収益企業(Q5)の総還元性向の分布（外れ値非表示）")

# 総還元性向の区分別の企業比率
bins = [-np.inf, 0.3, 0.5, 0.8, 1.0, np.inf]
names = ["30%未満", "30-50%", "50-80%", "80-100%", "100%以上"]
dist = (hp.assign(b=pd.cut(hp["total_payout_ratio"], bins, labels=names))
        .groupby("country")["b"].value_counts(normalize=True).unstack().reindex(["US", "JP"]))
left = np.zeros(2)
cmap = plt.get_cmap("Greens")
for i, n in enumerate(names):
    axes[1].barh(["米国", "日本"], dist[n], left=left, color=cmap(0.25 + 0.15 * i), label=n)
    for j, v in enumerate(dist[n]):
        if v > 0.05:
            axes[1].text(left[j] + v / 2, j, f"{v:.0%}", ha="center", va="center", fontsize=9)
    left += dist[n].values
axes[1].set_title("高収益企業(Q5)の総還元性向の構成")
axes[1].xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axes[1].legend(ncol=5, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout()
plt.show()
dist.rename(index=COUNTRY_LABELS)

In [ ]:
# 時系列：高収益企業とその他の総還元性向・総還元利回り
def plot_group_ts(data, cols, title):
    cols = [c for c in cols if c in data.columns]
    med = data.dropna(subset=["group"]).groupby(["date", "country", "group"])[cols].median()
    fig, axes = plt.subplots(1, len(cols), figsize=(6 * len(cols), 3.8), squeeze=False)
    for ax, col in zip(axes.flat, cols):
        for c in ["US", "JP"]:
            for g, ls in [("高収益(Q5)", "-"), ("その他(Q1-4)", "--")]:
                try:
                    s = med[col].xs((c, g), level=("country", "group")).rolling(12, min_periods=6).mean()
                except KeyError:
                    continue
                ax.plot(s.index, s, color=COUNTRY_COLORS[c], ls=ls, label=f"{COUNTRY_LABELS[c]}:{g}")
        ax.set_title(lab(col))
    axes.flat[0].legend(fontsize=8)
    fig.suptitle(title + "（中央値, 12M平均）", y=1.02)
    plt.tight_layout()
    plt.show()


plot_group_ts(df, ["total_payout_ratio", "gross_payout_yield"], "高収益企業とその他の総還元の推移")

In [ ]:
# 収益性と還元の連動：国×月ごとに『ROEの順位』と『総還元性向の順位』の相関（スピアマン）
#   米国で高く日本で低ければ、米国の方が「稼ぐ企業ほど返す」構造
rk = df.dropna(subset=[HIGH_PROF_VAR, "total_payout_ratio"])
rk = rk.assign(
    r_prof=rk.groupby(["country", "date"])[HIGH_PROF_VAR].rank(pct=True),
    r_pay=rk.groupby(["country", "date"])["total_payout_ratio"].rank(pct=True),
)
corr = rk.groupby(["date", "country"])[["r_prof", "r_pay"]].corr().xs("r_prof", level=2)["r_pay"].unstack()

fig, ax = plt.subplots(figsize=(12, 3.8))
for c in ["US", "JP"]:
    ax.plot(corr.index, corr[c].rolling(12, min_periods=6).mean(), color=COUNTRY_COLORS[c],
            label=f"{COUNTRY_LABELS[c]}（全期間平均 {corr[c].mean():+.2f}）")
ax.axhline(0, color="black", lw=0.8)
ax.set_title(f"{lab(HIGH_PROF_VAR)}と総還元性向の順位相関（12M平均）")
ax.legend()
plt.show()

### 11. バリュエーション：市場は高収益企業を評価しているか

In [ ]:
_ = plot_quintile_profiles(df, ["book_to_price_actual", "earnings_to_price_actual"], "バリュエーション", ncols=2)

# 高収益企業のB/P ÷ ユニバース中央値B/P（1未満＝割高に評価されている）
bp = df.dropna(subset=["q", "book_to_price_actual"])
rel = (bp[bp["is_high"] == True].groupby(["date", "country"])["book_to_price_actual"].median()
       / bp.groupby(["date", "country"])["book_to_price_actual"].median()).unstack()
fig, ax = plt.subplots(figsize=(12, 3.8))
for c in ["US", "JP"]:
    ax.plot(rel.index, rel[c], color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
ax.axhline(1, color="black", lw=0.8)
ax.set_title("高収益企業のB/P（中央値）÷ 全体のB/P（中央値）：低いほど市場がプレミアムを付与")
ax.legend()
plt.show()

B/P は「ROE × B/P = E/P」の関係で ROE と機械的に結び付くため、Q5 で B/P が低いのはある程度自然です。
注目点は **プレミアムの大きさの日米差** と **その推移**（例：日本で東証のPBR改善要請後に変化したか）。

### 12. 規模・成長・投資・財務構成

In [ ]:
_ = plot_quintile_profiles(
    df,
    ["log_mcap", "sales_growth_actual", "asset_growth_actual", "investment_to_assets_actual", "equity_ratio", "debt_to_assets"],
    "規模・成長・投資・財務構成",
)

## 13. 多変量分析：何が高収益企業を特徴づけるか（Fama-MacBeth型）

単変量では相関し合う特徴（例：自己資本比率と負債比率、規模と還元）の効果が混ざるため、
各国・各月で「高収益企業ダミー（Q5=1）」を自国内で標準化した特徴量に回帰（線形確率モデル）し、
係数の時系列平均と Newey-West t値を求めます。

ROE の定義に直接入る純利益率・ROA などは説明変数から外しています（B/P・回転率は ROE と機械的関係があるので解釈注意）。

In [ ]:
FM_FEATS = [c for c in [
    "log_mcap", "equity_ratio", "asset_turnover_actual", "gross_margin_actual",
    "sales_growth_actual", "asset_growth_actual", "book_to_price_actual",
    "gross_payout_yield",
] if c in df.columns]


def fama_macbeth_lpm(data, y_col, feats, min_obs=30):
    out = []
    for (c, d), g in data.dropna(subset=[y_col] + feats).groupby(["country", "date"]):
        if len(g) < max(min_obs, len(feats) + 10):
            continue
        X = g[feats]
        X = (X - X.mean()) / X.std()
        if X.isna().any().any():
            continue
        X = np.column_stack([np.ones(len(X)), X.to_numpy()])
        y = g[y_col].astype(float).to_numpy()
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        out.append(pd.Series(beta[1:], index=feats, name=(c, d)))
    coefs = pd.DataFrame(out)
    coefs.index = pd.MultiIndex.from_tuples(coefs.index, names=["country", "date"])
    res = pd.DataFrame({
        "coef": coefs.groupby("country").mean().stack(),
        "t": coefs.groupby("country").agg(lambda s: nw_tstat(s)).stack(),
    })
    return res, coefs


fm, fm_coefs = fama_macbeth_lpm(df, "is_high", FM_FEATS)
fm_tbl = fm.unstack("country")
fm_tbl.index = [lab(i) for i in fm_tbl.index]
fm_tbl

In [ ]:
fig, ax = plt.subplots(figsize=(9, 0.5 * len(FM_FEATS) + 1.5))
order = fm.xs("JP", level="country")["coef"].sub(fm.xs("US", level="country")["coef"]).sort_values().index
y = np.arange(len(order))
for c, off in (("US", 0.18), ("JP", -0.18)):
    r = fm.xs(c, level="country").reindex(order)
    se = r["coef"] / r["t"]
    ax.errorbar(r["coef"], y + off, xerr=1.96 * se.abs(), fmt="o", color=COUNTRY_COLORS[c],
                label=COUNTRY_LABELS[c], capsize=3)
ax.set_yticks(y)
ax.set_yticklabels([lab(i) for i in order])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("係数（特徴量1σ上昇あたりの高収益企業になる確率の変化）")
ax.set_title("高収益企業ダミーの月次クロスセクション回帰（95%区間, Newey-West）")
ax.legend()
plt.tight_layout()
plt.show()

## 14. 頑健性チェック

### 14-1. 高収益企業の定義を変える
ROE（基本）／ROA／収益性ファクター／ROE（セクター中立）の4定義で、日米差（JP−US）が同じ方向かを確認します。

In [ ]:
alt_defs = {}
for var in [v for v in ["roe_actual", "roa_actual", "profitability_factor"] if v in df.columns]:
    tmp = df.copy()
    tmp["flag"] = (assign_groups(tmp, var, ["country", "date"]) == HIGH_GROUP).where(tmp[var].notna())
    alt_defs[lab(var)] = high_group_profile(tmp, "flag")[0]["JP−US"]
alt_defs[f"{lab(HIGH_PROF_VAR)}（セクター中立）"] = profile_sec["JP−US"]
alt = pd.DataFrame(alt_defs)
alt = alt.drop(index=[c for c in FEATURE_BLOCK["収益性"] if c in alt.index]).sort_values(alt.columns[0])


def heatmap(mat, title, vmax=1.0, fmt="{:.2f}", figsize=None):
    figsize = figsize or (1.6 * mat.shape[1] + 4, 0.38 * mat.shape[0] + 1.6)
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(mat.values, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.values[i, j]
            if pd.notna(v):
                ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=8,
                        color="white" if abs(v) > 0.6 * vmax else "black")
    ax.set_xticks(range(mat.shape[1])); ax.set_xticklabels(mat.columns, rotation=30, ha="right")
    ax.set_yticks(range(mat.shape[0])); ax.set_yticklabels([lab(i) for i in mat.index])
    ax.set_title(title)
    ax.grid(False)
    plt.colorbar(im, ax=ax, shrink=0.7)
    plt.tight_layout()
    plt.show()


heatmap(alt, "定義別：高収益企業の特徴の日米差（JP−US, zスコア）")

### 14-2. 期間を分ける

In [ ]:
per = {}
for name, (y0, y1) in PERIODS.items():
    sub = df[(df["date"].dt.year >= y0) & (df["date"].dt.year <= y1)]
    if sub.empty:
        continue
    r, _ = high_group_profile(sub)
    per[(name, "米国")] = r["US_z"]
    per[(name, "日本")] = r["JP_z"]
per = pd.DataFrame(per)
per.columns = [f"{a}\n{b}" for a, b in per.columns]
heatmap(per, "期間別：高収益企業の平均zスコア", vmax=1.2)

## 15. まとめ（自動抽出）

以下は上の結果から機械的に抽出した要点です。数値を見ながら解釈を加えてください。

In [ ]:
def describe_findings(profile, share_mean, ratio, stay, fm, top=6):
    lines = []
    p = profile.drop(index=[c for c in FEATURE_BLOCK["収益性"] if c in profile.index]).copy()
    p["abs_gap"] = p["JP−US"].abs()
    lines.append("■ 高収益企業の特徴で日米差が大きい項目（自国内zスコアの差）")
    for f, r in p.sort_values("abs_gap", ascending=False).head(top).iterrows():
        side = "日本" if r["JP−US"] > 0 else "米国"
        lines.append(
            f"  - {lab(f)}: 米国 {r['US_z']:+.2f} (t={r['US_t']:.1f}) / 日本 {r['JP_z']:+.2f} (t={r['JP_t']:.1f})"
            f" → {side}の高収益企業でより高い傾向")

    lines.append("\n■ 日米で符号が逆（片方で高く、片方で低い）かつ双方有意な項目")
    opp = p[(np.sign(p["US_z"]) != np.sign(p["JP_z"])) & (p["US_t"].abs() > 2) & (p["JP_t"].abs() > 2)]
    lines += [f"  - {lab(f)}: 米国 {r['US_z']:+.2f} / 日本 {r['JP_z']:+.2f}" for f, r in opp.iterrows()] or ["  - なし"]

    lines.append("\n■ ROE格差の源泉（DuPont分散分解, 全期間平均）")
    for c in share_mean.index:
        s = share_mean.loc[c].drop("その他（近似誤差）")
        lines.append(f"  - {c}: " + ", ".join(f"{k} {v:.0%}" for k, v in s.items()) + f" → 主因は{s.idxmax()}")

    lines.append("\n■ 高収益企業（Q5）がその他に対して何倍か（DuPont構成要素）")
    for c in ratio.index:
        lines.append(f"  - {c}: " + ", ".join(f"{k} {v:.2f}倍" for k, v in ratio.loc[c].items()))

    lines.append("\n■ 持続性（Q5に留まる確率）")
    for h in stay.index:
        lines.append(f"  - {h}か月後: 米国 {stay.loc[h, 'US']:.0%} / 日本 {stay.loc[h, 'JP']:.0%}")

    lines.append("\n■ 多変量回帰で有意（|t|>2）な特徴")
    for c in ["US", "JP"]:
        r = fm.xs(c, level="country")
        sig = r[r["t"].abs() > 2].sort_values("t")
        lines.append(f"  - {COUNTRY_LABELS[c]}: " + (", ".join(f"{lab(i)}({v:+.3f})" for i, v in sig["coef"].items()) or "なし"))
    return "\n".join(lines)


print(f"定義: {lab(HIGH_PROF_VAR)} の国内上位{100 // N_GROUPS}%  /  除外セクター: {EXCLUDE_SECTOR_CODES}  /  "
      f"期間: {df['date'].min():%Y-%m}〜{df['date'].max():%Y-%m}\n")
print(describe_findings(profile, share_mean, ratio, stay, fm))

### 解釈の観点（検証すべき仮説）

結果を読む際の主なチェックポイントです。上の出力でどれが支持されるかを確認してください。

1. **稼ぎ方の違い**：日本の高収益企業は「マージン型」か「回転率型」か。米国はレバレッジ（自己資本比率の低さ）がROEを押し上げているか
   → 6章（DuPont）と5章の自己資本比率・負債比率
2. **財務の保守性**：日本の高収益企業は自国内でも自己資本比率が高い（キャッシュリッチ）のか。そうならROEを抑える要因になっている可能性
3. **稼いだ利益の還元**：米国の高収益企業は利益の大半（総還元性向が高い）を株主に返し、日本の高収益企業は内部留保に回しているか。
   日本で近年その差が縮まっているか → 10章の総還元性向
4. **市場の評価**：高収益企業へのバリュエーション・プレミアムは日米どちらが大きいか、日本で拡大しているか → 11章
5. **持続性**：日本の高収益企業の方が入れ替わりが激しい（景気敏感・市況依存）か → 8章
6. **業種要因**：日米差が業種構成（米国はIT・ヘルスケア、日本は資本財・一般消費財など）で説明されるか → 7章のセクター中立比較
7. **成長との関係**：高収益企業は成長企業でもあるのか（米国）、成熟企業なのか（日本） → 12章